# One-star simulation through multiple atmospheric phase screens

This notebook generates a mock guider movie using `GalSim`. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from tqdm.auto import tqdm
import galsim

print("GalSim version:", galsim.__version__)

GalSim version: 2.8.5


## Simulation parameters

In [2]:
# Timing
cadence = 0.5       # seconds between frame starts
exptime = 0.05      # seconds integrated in each frame
duration = 15.0     # seconds
frame_times = np.arange(0.0, duration, cadence)
n_frames = len(frame_times)

# Telescope and detector
diam = 8.36
obscuration = 0.612
lam = 700.0         # nm
pixel_scale = 0.2   # arcsec / pixel
stamp_size = 32     # pixels

# Atmosphere
r0_500_total = 0.10
L0 = 25.0

altitudes = np.array([0.2, 5.0, 12.0])     # km
weights = np.array([0.60, 0.25, 0.15])
wind_speeds = np.array([5.0, 10.0, 18.0])  # m/s
wind_directions = np.array([0.0, 55.0, 120.0]) * galsim.degrees

weights = weights / weights.sum()
r0_500_layers = r0_500_total * weights**(-3.0 / 5.0)

# Phase-screen sampling. Increase screen_size for longer runs or faster winds.
screen_size = 128.0  # m
screen_scale = 0.05  # m

# photon shooting
n_photons = 200_000
seed = 20241103

print(f"{n_frames} frames from t={frame_times[0]:.1f} to {frame_times[-1]:.1f} s")
print("Per-layer r0_500 [m]:", np.round(r0_500_layers, 3))
print(f"Photons per photon-shot frame: {n_photons:,}")


30 frames from t=0.0 to 14.5 s
Per-layer r0_500 [m]: [0.136 0.23  0.312]
Photons per photon-shot frame: 200,000


## Construct the frozen-flow atmosphere

In [3]:
seed = 12345

atm_fft = galsim.Atmosphere(
    r0_500=r0_500_layers,
    altitude=altitudes,
    speed=wind_speeds,
    direction=wind_directions,
    screen_size=screen_size,
    screen_scale=screen_scale,
    rng=galsim.BaseDeviate(seed),
)

atm_phot = galsim.Atmosphere(
    r0_500=r0_500_layers,
    altitude=altitudes,
    speed=wind_speeds,
    direction=wind_directions,
    screen_size=screen_size,
    screen_scale=screen_scale,
    rng=galsim.BaseDeviate(seed),
)

aperture = galsim.Aperture(
    diam=diam,
    obscuration=obscuration,
    lam=lam,
)

## Simulate and store all frames

`t0` is the start time of each exposure. With photon shooting and the default frozen-flow atmosphere, photons sample the moving screens continuously over each 0.05 s exposure.

In [4]:
fft_images = np.empty(
    (n_frames, stamp_size, stamp_size),
    dtype=np.float32,
)
phot_images = np.empty_like(fft_images)

photon_rng = galsim.BaseDeviate(seed + 1)

for i, t0 in enumerate(
    tqdm(frame_times, desc="Rendering FFT and photon frames")
):
    psf_fft = atm_fft.makePSF(
        lam=lam,
        aper=aperture,
        t0=float(t0),
        exptime=exptime,
        time_step=0.005,
    )

    fft_image = psf_fft.drawImage(
        nx=stamp_size,
        ny=stamp_size,
        scale=pixel_scale,
        method="fft",
    )

    psf_phot = atm_phot.makePSF(
        lam=lam,
        aper=aperture,
        t0=float(t0),
        exptime=exptime,
        geometric_shooting=True,
    )

    phot_image = psf_phot.drawImage(
        nx=stamp_size,
        ny=stamp_size,
        scale=pixel_scale,
        method="phot",
        n_photons=n_photons,
        poisson_flux=False,
        rng=photon_rng,
    )

    fft_images[i] = fft_image.array
    phot_images[i] = phot_image.array

Rendering FFT and photon frames:   0%|          | 0/30 [00:00<?, ?it/s]

## Animate the stored frames

In [5]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Choose one:
frames = fft_images
# frames = photon_images

fig, ax = plt.subplots(figsize=(5, 5))

vmin, vmax = 0, np.percentile(frames, 99.9)

im = ax.imshow(
    frames[0],
    origin="lower",
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)

title = ax.set_title("Frame 0")
ax.set_xticks([])
ax.set_yticks([])
fig.colorbar(im, ax=ax, label="Flux")

def update(i):
    im.set_data(frames[i])
    title.set_text(f"Frame {i} — t = {i * cadence:.1f} s")
    return im, title

anim = FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=cadence * 500,
    blit=True,
)

plt.close(fig)
HTML(anim.to_jshtml())

In [6]:
# comparison of the two
vmax = np.percentile(np.concatenate([fft_images.ravel(), phot_images.ravel()]), 99.9)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharex=True, sharey=True)

fft_artist = axes[0].imshow(
    fft_images[0], origin="lower", vmin=0, vmax=vmax, cmap="magma", animated=True
)
phot_artist = axes[1].imshow(
    phot_images[0], origin="lower", vmin=0, vmax=vmax, cmap="magma", animated=True
)

axes[0].set_title("FFT")
axes[1].set_title(f"Photon shooting ({n_photons:,} photons)")
for ax in axes:
    ax.set_xlabel("x [pixel]")
axes[0].set_ylabel("y [pixel]")

time_title = fig.suptitle(f"Same atmospheric PSF | t = {frame_times[0]:.1f} s")
fig.colorbar(fft_artist, ax=axes, fraction=0.035, pad=0.03, label="Flux / pixel")


def update_comparison(frame_index):
    fft_artist.set_data(fft_images[frame_index])
    phot_artist.set_data(phot_images[frame_index])
    time_title.set_text(f"Same atmospheric PSF | t = {frame_times[frame_index]:.1f} s")
    return fft_artist, phot_artist, time_title


comparison_animation = FuncAnimation(
    fig,
    update_comparison,
    frames=n_frames,
    interval=500 * cadence,
    blit=False,
)
plt.close(fig)

HTML(comparison_animation.to_jshtml())
